# Feature Engineering — Airline Passenger Satisfaction (Kaggle Playground S6E10)

**Task:** binary classification (`satisfaction` = satisfied / not satisfied). **Metric:** ROC-AUC.

**Purpose of this notebook:** try new features on top of the baseline model and keep only those that improve the
cross-validated AUC. Every idea is tested the same way, written down with *where the idea came from*, and closed
with a short conclusion. The final result is a small, justified feature set that is exported for the modelling stage.

**Headline result:** baseline CV AUC **0.95885** → CV AUC **0.96015** (**+0.00130**) with only **3 new features**: two "how often does this value appear" counts (round 1) and one target encoding of `flight_distance` computed inside each fold (round 2). Everything else we tried did not help.

> **How to read this notebook.** Sections 1–4 are setup and the testing protocol. Sections 5–15 are data checks and experiments (E1–E9), each
> with *Idea origin → Code → Conclusion*. Section 16 builds the final feature set of round 1, section 17 tests the original
> dataset, section 18 is **round 2** (experiments E10–E15 based on ideas from public notebooks) and section 19 is the summary. > A full *Run All* takes roughly 25–30 minutes (about 35 seconds per CV run).

---

## Table of contents

| # | Section | What it answers |
|---|---|---|
| 1 | [Setup & data loading](#s1) | Imports, paths, reading the files |
| 2 | [Preprocessing (same as baseline)](#s2) | How `X`, `X_test`, `y` are built |
| 3 | [Evaluation framework](#s3) | The CV function and the "is it signal or noise?" rule |
| 4 | [How we come up with feature ideas](#s4) | The thinking process behind the experiments |
| 5 | [Look at the data first: rating zeros](#s5) | Why a `0` rating needs special care |
| 6 | [E1 — Aggregates of the ratings](#e1) | mean / min / max / std / counts per passenger |
| 7 | [E2 — Delay difference](#e2) | arrival delay minus departure delay |
| 8 | [E3 — Rating relative to the passenger's class](#e3) | group mean and deviation from it |
| 9 | [E4 — Passenger segment](#e4) | class × type of travel × customer type as one category |
| 10 | [Analysis — which aspects matter?](#s10) | Mann-Whitney effect sizes and LightGBM importance |
| 11 | [E5 — Arithmetic between the top ratings](#e5) | sum / difference / product of strongest ratings |
| 12 | [E6 — Removing weak features](#e6) | does less noise help? |
| 13 | [E7 — Ratings as categories](#e7) | 0–5 as labels instead of numbers |
| 14 | [E8 — Frequency features](#e8) | **the idea that worked** |
| 15 | [E9 — Frequency of value combinations](#e9) | adding `type_of_travel` to the distance count |
| 16 | [Final feature set, verification and export](#s16) | `add_features()` and the saved CSV files |
| 17 | [Using the original dataset](#s17) | 4 ways to use the source data, none helped |
| 18 | [Round 2 — ideas from public notebooks](#s18) | E10–E15: bins, digits, **target encoding**, distance profile |
| 19 | [Summary](#s19) | Results table, chart, lessons, next steps |

## Feature map — every feature we considered

| ID | Feature(s) | Idea source | Result vs baseline | Verdict |
|---|---|---|---|---|
| E1 | `rating_mean`, `rating_min`, `rating_max`, `rating_std`, `n_low`, `n_high`, `n_zero_digital` | Standard technique (row-wise aggregation) | −0.00021 | Rejected |
| E2 | `delay_diff` | Domain knowledge (was the delay made worse in the air?) | −0.00001 | Rejected |
| E3 | `ob_class_mean`, `ob_class_diff` | Standard technique (group-relative feature) | 0.00000 / −0.00003 | Rejected |
| E4 | `segment` | EDA (hypotheses about Business / loyal / personal travel groups) | +0.00008 | Noise level, not taken |
| E5 | sum / difference / product of the top ratings | Standard technique (interactions) | −0.00008 … −0.00012 | Rejected |
| E6 | (removal of `arrival_delay_missing`, `gender`, `departure_delay_in_minutes`) | Feature-importance analysis | −0.00002 … −0.00007 | Keep all columns |
| E7 | ratings as `category` | Observation that rating `0` is special | −0.00013 / −0.00004 | Rejected |
| **E8** | **`flight_distance_freq`** | Observation: rare distances ↔ more satisfied | **+0.00059** | **Accepted** |
| **E9** | **`flight_distance_type_of_travel_freq`** | Extension of E8 (count of a combination) | **+0.00020** on top of E8 | **Accepted** |
| §17 | `orig_pred`, appended original rows | Source dataset | −0.00008 … −0.00035 | Rejected |
| E10 | `fd_div10`, `fd_div100`, `fd_mod1000` | Public notebook | −0.00001 … −0.00012 (vs round-1 set) | Rejected |
| E11 | digits of `flight_distance`, `age`, `departure_delay` | Public notebook | −0.00022 … +0.00006 | Rejected |
| **E12** | **`te_flight_distance`** (in-fold target encoding) | Public notebook | **+0.00051** | **Accepted** |
| E13 | TE of `age`, distance × type / class | Extension of E12 | +0.00001 … +0.00011 (vs E12) | Not taken |
| E14 | per-distance profile of 3 ratings | Public notebook | ±0.00000 (vs E12) | Rejected |
| E15 | TE of distance / age bins | Public notebook | −0.00006 … +0.00002 (vs E12) | Rejected |

<a id="s1"></a>
## 1. Setup & data loading

Imports, paths and reading the competition files. Column names are normalised to `lower_case_with_underscores`
so that we can type them without worrying about spaces and capital letters.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Move to the project folder (local machine). On Kaggle this folder does not exist, so nothing happens.
PROJECT_DIR = Path('/Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10')
if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)

# Raw data: Kaggle path if we are on Kaggle, otherwise the local folder
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if os.path.exists('/kaggle/input') else Path('data/raw')
PROC = Path('data/processed')      # where the final feature tables are saved

print("Imports OK")
print(f"Working directory: {os.getcwd()}")
print(f"Data path: {DATA_DIR}")

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

# 'Inflight wifi service' -> 'inflight_wifi_service'
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

<a id="s2"></a>
## 2. Preprocessing (identical to `baseline.ipynb`)

We start from **exactly the data the baseline model used** — this is the only honest starting point: any improvement
we measure must come from the new features, not from a different preparation of the data.

Steps:
1. `arrival_delay_missing` — a 0/1 flag marking rows where `arrival_delay_in_minutes` was empty. Created **before**
   filling, otherwise the information about the gap is lost.
2. Fill the empty `arrival_delay_in_minutes` with `departure_delay_in_minutes` (the two columns are almost the same, correlation 0.97).
3. Target `y`: `True/False` → `1/0`.
4. Drop `id`, `satisfaction` (it is the target) and `arrival_delay_in_minutes` (a near-duplicate of the departure delay).
5. Mark the four text columns as pandas `category` so LightGBM treats them as categories.

Result: **21 features** — this is our "starting set" called `BASE_COLS` later on.

In [ ]:
# 1. Missing-value flag - BEFORE fillna, while the information about the gap still exists
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)

# 2. Fill the gaps in arrival_delay with the departure delay
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(
    train['departure_delay_in_minutes']
)
test['arrival_delay_in_minutes'] = test['arrival_delay_in_minutes'].fillna(
    test['departure_delay_in_minutes']
)

# 3. Target: True/False -> 1/0
y = train['satisfaction'].astype(int)

# 4. Columns to drop and the list of categorical columns
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']

# 5. Build X and X_test
X      = train.drop(columns=DROP_COLS)
X_test = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# 6. Categorical columns -> pandas 'category' type (needed by LightGBM)
for col in CAT_COLS:
    X[col]      = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Features: {len(X.columns)}")

The 21 baseline features fall into four groups (useful later when we think about *which columns could be combined*):

| Group | Features |
|---|---|
| Passenger context | `age`, `gender`, `customer_type`, `type_of_travel` |
| Trip context | `class`, `flight_distance` |
| Service ratings (0–5) | `inflight_wifi_service`, `online_boarding`, `food_and_drink`, `seat_comfort`, `inflight_entertainment`, `on-board_service`, `leg_room_service`, `baggage_handling`, `checkin_service`, `cleanliness`, `gate_location`, `departure/arrival_time_convenient`, `ease_of_online_booking` |
| Operational | `departure_delay_in_minutes`, `arrival_delay_missing` |

<a id="s3"></a>
## 3. Evaluation framework

**One protocol for every experiment**, so that results are comparable:

* Model: LightGBM with fixed parameters (the same as in the baseline).
* Validation: 5-fold `StratifiedKFold(shuffle=True, random_state=42)` — every experiment sees exactly the same folds.
* Score: ROC-AUC computed on the **out-of-fold (OOF)** predictions, i.e. on rows the model never saw during training.
* Comparison: always against the baseline score recomputed *in the same notebook* (`BASELINE_CV = 0.95885`).

**Is a difference real or noise?** Re-running the same model gives differences of about 0.00003–0.0001 (LightGBM uses
several threads), and the fold-to-fold standard deviation is about ±0.0006. Therefore:

| Change in CV AUC | Interpretation |
|---|---|
| below about 0.0001 | noise |
| from +0.0003, and/or the same sign on all 5 folds | signal |
| negative | the feature hurts (adds noise) |

In [ ]:
LGBM_PARAMS = {
    'objective':        'binary',
    'metric':           'auc',
    'learning_rate':    0.05,
    'num_leaves':       127,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'n_estimators':     1000,
    'early_stopping_rounds': 50,   # stop when the validation AUC has not improved for 50 trees
    'verbose':          -1,
    'random_state':     42,
}

N_FOLDS = 5

In [ ]:
def run_cv(X, y, X_test, cat_cols):
    """Train LightGBM on 5 folds. Returns CV AUC, out-of-fold predictions and test predictions."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    oof_preds   = np.zeros(len(X))       # prediction for every train row, made by a model that did not see it
    test_preds  = np.zeros(len(X_test))  # test predictions averaged over the 5 fold-models
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr,  X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr,  y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**LGBM_PARAMS)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            categorical_feature=cat_cols,
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"Fold {fold} AUC: {score:.5f}")

    cv_score = roc_auc_score(y, oof_preds)
    print(f"\nCV AUC (OOF): {cv_score:.5f}")
    print(f"Mean fold AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

    return cv_score, oof_preds, test_preds

**Baseline run.** We train on the 21 baseline features. The result becomes the reference for all experiments.
(The separate `baseline.ipynb` gave 0.95882; the 0.00003 difference is the multithreading noise mentioned above.)

In [ ]:
baseline_score, _, _ = run_cv(X, y, X_test, CAT_COLS)

In [ ]:
BASELINE_CV = 0.95885     # the baseline score recorded when we ran the cell above

**Helper for the experiments.** Every experiment repeats the same pattern: "take a table, choose columns, run CV,
print the gain versus the baseline". `evaluate()` does it in one call.

`run_cv` needs a `X_test` argument, but during feature testing we only care about the out-of-fold score, so we pass a
small stub (the first 1000 rows of the training table) to keep things fast.

In [ ]:
def evaluate(df, cols, cat_cols, label, ref=BASELINE_CV, ref_name='baseline'):
    """Run CV on df[cols] and print the score and the gain versus a reference score."""
    stub = df[cols].iloc[:1000]                        # stand-in for X_test (not used for scoring here)
    score, _, _ = run_cv(df[cols], y, stub, cat_cols)
    print(f">>> {label}: CV AUC {score:.5f} | vs {ref_name} {score - ref:+.5f}\n")
    return score

<a id="s4"></a>
## 4. How we come up with feature ideas

**Where ideas come from** (we used all three):

1. **Domain knowledge** — how does the real world work? (Example: "a flight that arrives later than it departed is worse
   than one that kept its delay" → `delay_diff`.)
2. **EDA findings** — what did we see in the data? (Example: rating `0` behaves strangely → special handling of zeros;
   rare flight distances go with more satisfied passengers → frequency features.)
3. **Standard techniques** — a checklist of what usually works on tabular data: aggregation, interaction, transformation,
   bucketing, flags, differences/ratios.

**The key question before testing any idea:** *does the new column contain information the model cannot already
get from the existing columns?*

* Gradient-boosted trees are good at logical combinations ("rating ≥ 4 **and** class = Business") and at shifts per segment,
  so features that merely repackage a single row's values usually add nothing.
* Trees are weak at **cross-column arithmetic** (sums, ratios) and have **no access to other rows**. Group statistics and
  counts (information about other rows) are therefore the most promising family.

**Two kinds of aggregation:**
* *Row-wise* — one number per passenger computed across columns (mean of all ratings, number of low ratings).
* *Group-by* — one number per group of passengers (mean `online_boarding` within each `class`) and the passenger's deviation from it.

**Order of work:** squeeze the most out of LightGBM features first, then reuse the best feature set for CatBoost / XGBoost.

<a id="s5"></a>
## 5. Look at the data first: rating zeros

Before building rating aggregates we must understand the value `0`. In this dataset ratings are 1–5, and `0` means
"not applicable / not rated". If we treat `0` as an ordinary low score, every average gets polluted.

Two questions:
1. In which columns are zeros common?
2. Is `0` the *worst* rating, or something else?

In [ ]:
RATING_COLS = [
    'inflight_wifi_service', 'departure/arrival_time_convenient', 'ease_of_online_booking',
    'gate_location', 'food_and_drink', 'online_boarding', 'seat_comfort',
    'inflight_entertainment', 'on-board_service', 'leg_room_service',
    'baggage_handling', 'checkin_service', 'cleanliness',
]

# How many zeros does each rating column have?
(X[RATING_COLS] == 0).sum().sort_values(ascending=False)

Zeros are concentrated in four columns: `departure/arrival_time_convenient` (25,751),
`ease_of_online_booking` (16,073), `inflight_wifi_service` (13,102) and `online_boarding` (9,147). Every other column has at most 1,693,
and `baggage_handling` has none. Three of the four are *digital* services (wifi, boarding, booking).

Now the satisfaction rate for each rating value, including `0`:

In [ ]:
for col in ['inflight_wifi_service', 'online_boarding',
            'ease_of_online_booking', 'departure/arrival_time_convenient']:
    print(col)
    print(train.groupby(col)['satisfaction'].agg(['mean', 'count']))
    print()

In [ ]:
# Chart: share of satisfied passengers for each rating value (0 to 5)
fig, ax = plt.subplots(figsize=(7, 4))
INK, MUTED, GRID = '#1f2937', '#6b7280', '#e5e7eb'
for col, color in [('inflight_wifi_service', '#2563eb'), ('online_boarding', '#d9822b'),
                   ('ease_of_online_booking', '#0f766e')]:
    rate = train.groupby(col)['satisfaction'].mean()
    ax.plot(rate.index, rate.values, marker='o', linewidth=2, color=color, label=col)
ax.set_xlabel('Rating value (0 = not rated)', color=MUTED)
ax.set_ylabel('Share of satisfied passengers', color=MUTED)
ax.set_title('Rating 0 is NOT the worst rating', color=INK, loc='left')
ax.grid(axis='y', color=GRID); ax.set_axisbelow(True)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### Conclusion — rating zeros

* A `0` is **not** "the worst score": passengers who gave `0` to `inflight_wifi_service` are 89% satisfied, `online_boarding` 62%, `ease_of_online_booking` 71%.
  They most likely *did not use the service*.
* `departure/arrival_time_convenient` is flat across all values — it barely relates to satisfaction at all.
* Satisfaction jumps between rating **3 and 4**.

**Decision for rating aggregates:** replace `0` with `NaN` before averaging (so `0` is simply ignored, and `mean/min/max/std`
skip it), and keep the information "how many digital services were skipped" in a separate counter (`n_zero_digital`).

<a id="e1"></a>
## 6. E1 — Aggregates of the ratings (row-wise)

### Idea origin
*Standard technique + domain thinking.* A passenger gives 13 ratings. Instead of making the model look at 13 numbers
separately, summarise them per passenger. Each aggregate answers a business question:

| Aggregate | Question it answers |
|---|---|
| `rating_mean` | How satisfied is the passenger overall? |
| `rating_min` | Is there at least one strong complaint? |
| `rating_max` | Is there something the passenger loved? |
| `rating_std` | Are the ratings even, or do they jump (everything fine but the wifi is terrible)? |
| `n_low` (rating ≤ 3) | How many things annoyed the passenger? |
| `n_high` (rating ≥ 4) | How many things did the passenger like? (the threshold 3/4 comes from the jump seen in section 5) |
| `n_zero_digital` | How many of the three digital services (wifi, boarding, booking) were skipped? |

### Method
Work on a copy `x2` (so the baseline table `X` stays untouched). Zeros are turned into `NaN` first, `axis=1` means
"compute across the columns of one row". We test all aggregates together, then by group (levels vs counters), then `n_zero_digital` alone.

In [ ]:
x2 = X.copy(deep=True)

# 0 means "not rated" -> turn into NaN so that mean/min/max/std ignore it
ratings = x2[RATING_COLS].replace(0, np.nan)

DIGITAL_COLS = ['inflight_wifi_service', 'online_boarding', 'ease_of_online_booking']

# axis=1 -> compute across columns, i.e. one value per passenger (row)
x2['rating_mean'] = ratings.mean(axis=1)
x2['rating_min']  = ratings.min(axis=1)
x2['rating_max']  = ratings.max(axis=1)
x2['rating_std']  = ratings.std(axis=1)
x2['n_low']       = (ratings <= 3).sum(axis=1)     # NaN comparisons are False, so skipped ratings are not counted
x2['n_high']      = (ratings >= 4).sum(axis=1)
x2['n_zero_digital'] = (x2[DIGITAL_COLS] == 0).sum(axis=1)   # counted on the ORIGINAL zeros

In [ ]:
# Sanity check: 21 original + 7 new = 28 columns
new_cols = ['rating_mean', 'rating_min', 'rating_max', 'rating_std', 'n_low', 'n_high', 'n_zero_digital']
print(x2.shape)               # expected (699635, 28)
x2[new_cols].describe()

In [ ]:
BASE_COLS  = list(X.columns)    # the original 21 features
LEVEL_COLS = ['rating_mean', 'rating_min', 'rating_max', 'rating_std']   # "level" aggregates
COUNT_COLS = ['n_low', 'n_high', 'n_zero_digital']                        # counters

# (a) all 7 new features
score_fe = evaluate(x2, list(x2.columns), CAT_COLS, 'E1a: all 7 aggregates')

# (b) levels only
score_level = evaluate(x2, BASE_COLS + LEVEL_COLS, CAT_COLS, 'E1b: level aggregates')

# (c) counters only
score_count = evaluate(x2, BASE_COLS + COUNT_COLS, CAT_COLS, 'E1c: counters')

# (d) n_zero_digital alone
score_nz = evaluate(x2, BASE_COLS + ['n_zero_digital'], CAT_COLS, 'E1d: n_zero_digital only')

### Conclusion — E1
| Variant | Change vs baseline |
|---|---|
| all 7 aggregates | −0.00021 |
| level aggregates (mean/min/max/std) | −0.00021 |
| counters (n_low, n_high, n_zero_digital) | −0.00009 |
| `n_zero_digital` alone | −0.00007 |

**Rejected.** Every variant is slightly *worse* than the baseline. A tree already sees all 13 ratings of the row and can
build any "how many ratings are low" logic itself, so a per-row summary adds noise, not information. The 21-column
set stays unchanged.

*Lesson:* a feature computed only from the same row's columns rarely helps a tree model.

<a id="e2"></a>
## 7. E2 — Delay difference

### Idea origin
*Domain knowledge.* Passengers react to delays. If the flight lands later than it left, the delay grew in the air; if
it lands earlier, the crew made up time. `arrival_delay − departure_delay` expresses that, and the difference between two columns
is exactly the kind of arithmetic trees do poorly. We use the filled `arrival_delay_in_minutes` from `train`.

In [ ]:
x3 = X.copy(deep=True)    # a fresh working copy for the next experiments

x3['delay_diff'] = train['arrival_delay_in_minutes'] - x3['departure_delay_in_minutes']
x3['delay_diff'].describe()

In [ ]:
score_dd = evaluate(x3, BASE_COLS + ['delay_diff'], CAT_COLS, 'E2: delay_diff')

### Conclusion — E2
Change vs baseline: **−0.00001** (noise). **Rejected.**
The difference between arrival and departure delay is almost always zero, so the model has nothing to learn from it.

<a id="e3"></a>
## 8. E3 — Rating relative to the passenger's class

### Idea origin
*Standard technique (group-by aggregation).* Business-class passengers rate things differently from Economy passengers.
A rating of 4 may be "good for Economy" but "average for Business". So we compute the **mean `online_boarding` rating
within each class** and the passenger's **deviation** from it. We choose `online_boarding` because it is by far the
strongest aspect (section 10). Zeros are ignored (`NaN`), as decided in section 5.

In [ ]:
ratings_3 = x3[RATING_COLS].replace(0, np.nan)    # ratings with 0 -> NaN (same logic as in E1)

# Mean online_boarding within each class, written to every row of that class
x3['ob_class_mean'] = ratings_3['online_boarding'].groupby(x3['class']).transform('mean')

# How far this passenger is above/below the typical rating of their class
x3['ob_class_diff'] = ratings_3['online_boarding'] - x3['ob_class_mean']

In [ ]:
# Sanity checks
print(x3['ob_class_mean'].nunique())                         # expected 3 (one value per class)
print(x3.groupby('class')['ob_class_diff'].mean())           # expected about 0 in every class
print(x3[['ob_class_mean', 'ob_class_diff']].isna().sum())   # diff: ~9,147 gaps (the zeros), mean: 0

In [ ]:
score_both = evaluate(x3, BASE_COLS + ['ob_class_mean', 'ob_class_diff'], CAT_COLS, 'E3a: class mean + diff')
score_diff = evaluate(x3, BASE_COLS + ['ob_class_diff'],                  CAT_COLS, 'E3b: diff only')

### Conclusion — E3
| Variant | Change vs baseline |
|---|---|
| class mean + deviation | 0.00000 |
| deviation only | −0.00003 |

**Rejected.** No gain, and the sign differs between folds — this is noise. The "mean per class" is just three numbers,
which the tree can already reproduce by splitting on `class`; and a shift per segment is something trees do themselves.

<a id="e4"></a>
## 9. E4 — Passenger segment

### Idea origin
*EDA hypotheses.* In EDA we suspected interactions between passenger groups: Business class on long trips, young people
on personal travel, loyal customers who tolerate delays. A direct way to give the model all combinations at once is a single
categorical column that glues three columns together, e.g. `Business_Business travel_Loyal Customer`
(at most 3 × 2 × 2 = 12 segments).

In [ ]:
# Glue three category columns into one string, e.g. "Business_Business travel_Loyal Customer"
x3['segment'] = (
    x3['class'].astype(str) + '_' +
    x3['type_of_travel'].astype(str) + '_' +
    x3['customer_type'].astype(str)
)

# LightGBM needs the 'category' type
x3['segment'] = x3['segment'].astype('category')

print(x3['segment'].nunique())   # expected: no more than 12

In [ ]:
# How big is each segment and how satisfied are its passengers?
check = pd.DataFrame({'count': x3['segment'].value_counts(),
                      'satisfied_rate': y.groupby(x3['segment']).mean()})
check.sort_values('count', ascending=False)

In [ ]:
cat_seg = CAT_COLS + ['segment']      # 'segment' is now a categorical column too
score_seg = evaluate(x3, BASE_COLS + ['segment'], cat_seg, 'E4: segment')

### Conclusion — E4
Change vs baseline: **+0.00008**, positive on all five folds, but far below the +0.0003 signal threshold — at the edge of noise.
**Not taken into the final set.** The model can already form these combinations from the three separate columns.

<a id="s10"></a>
## 10. Analysis — which aspects actually matter?

This is not a feature test but a *map of where the signal lives*. It tells us which ratings are worth building
interactions from (E5) and which columns could be removed (E6).

### 10.1 Effect size per rating (Mann-Whitney U)
For each rating we compare the values of satisfied and not-satisfied passengers.

* The p-value is useless here (with 700,000 rows everything is "significant"), so we look at the **effect size**:
  rank-biserial correlation `r = 1 − 2U / (n1 · n2)`.
* **Direction:** `scipy` returns `U` for the *first* group. We pass `satisfied` first, so **negative `r` means satisfied passengers give higher ratings**.
* Zeros are excluded (`NaN` dropped).

In [ ]:
from scipy import stats

results = []

for col in RATING_COLS:
    data   = ratings_3[col].dropna()          # ratings without the zeros
    target = y.loc[data.index]                # the target for the same rows

    satisfied     = data[target == 1]
    not_satisfied = data[target == 0]

    # U statistic: U belongs to the FIRST group (satisfied)
    stat, p_value = stats.mannwhitneyu(satisfied, not_satisfied, alternative='two-sided')

    n_1 = len(satisfied)
    n_2 = len(not_satisfied)
    r = 1 - (2 * stat) / (n_1 * n_2)          # rank-biserial effect size

    results.append({
        'feature': col,
        'r': r,
        'p_value': p_value,
        'higher_in': 'satisfied' if r < 0 else 'not_satisfied',
    })

results_df = pd.DataFrame(results)
results_df.sort_values('r', key=abs, ascending=False)

Recorded result (ordered by strength of effect):

| Strength | Aspects (r) |
|---|---|
| Strong | `online_boarding` (−0.71), `inflight_entertainment` (−0.49), `seat_comfort` (−0.46), `on-board_service` (−0.40), `cleanliness` (−0.38), `leg_room_service` (−0.38), `inflight_wifi_service` (−0.37), `baggage_handling` (−0.31) |
| Medium | `checkin_service` (−0.27), `ease_of_online_booking` (−0.25), `food_and_drink` (−0.24) |
| Negligible | `departure/arrival_time_convenient` (+0.05), `gate_location` (−0.03) |

### 10.2 What does the model itself use? (LightGBM gain importance)
Gain importance = how much each feature improved the model's loss across all splits. It is biased towards
columns with many distinct values, so we use it as a guide, not as proof. The model is trained once on all data with 300 trees.

In [ ]:
params = {**LGBM_PARAMS, 'n_estimators': 300, 'early_stopping_rounds': None}   # no validation set -> no early stopping

model = lgb.LGBMClassifier(**params, importance_type='gain')
model.fit(X, y, categorical_feature=CAT_COLS)

imp = pd.Series(model.feature_importances_, index=X.columns)
imp_share = (imp / imp.sum()).sort_values(ascending=False)   # share of total gain, sums to 1
imp_share

In [ ]:
# Chart: share of total gain per feature (top 10)
top = imp_share.head(10)[::-1]                     # reversed, so the biggest bar is on top
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index, top.values * 100, color='#2563eb', height=0.6)
for i, v in enumerate(top.values * 100):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', color='#1f2937', fontsize=9)
ax.set_xlabel('Share of total gain, %', color='#6b7280')
ax.set_title('Signal is concentrated in a few features', color='#1f2937', loc='left')
ax.grid(axis='x', color='#e5e7eb'); ax.set_axisbelow(True)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

### Conclusion — analysis
* The signal is concentrated: `online_boarding` ≈ 43.7% of the gain, `inflight_wifi_service` ≈ 14.3%, `class` ≈ 10.5%,
  `type_of_travel` ≈ 7.6%, `inflight_entertainment` ≈ 5.0%. The five columns carry about 81% of all the gain.
* `arrival_delay_missing` ≈ 0, `gender` ≈ 0.1%, delays ≈ 0.15% — almost unused.
* The three strongest ratings (`online_boarding`, `inflight_wifi_service`, `inflight_entertainment`) are the natural candidates for E5.

<a id="e5"></a>
## 11. E5 — Arithmetic between the top ratings

### Idea origin
*Standard technique (interactions) guided by the analysis in section 10.* Trees split on one column at a time, so
they are weak at "A + B", "A − B" and "A × B". We test the three pairs built from the three strongest ratings:
sum, difference, product for each pair (9 candidate features).

In [ ]:
pairs = [
    ('online_boarding',       'inflight_wifi_service'),
    ('online_boarding',       'inflight_entertainment'),
    ('inflight_wifi_service', 'inflight_entertainment'),
]

for a, b in pairs:
    x3[f'{a}_plus_{b}']  = ratings_3[a] + ratings_3[b]
    x3[f'{a}_minus_{b}'] = ratings_3[a] - ratings_3[b]
    x3[f'{a}_times_{b}'] = ratings_3[a] * ratings_3[b]

In [ ]:
# Test each pair separately (3 new columns at a time)
for a, b in pairs:
    new_cols = [f'{a}_plus_{b}', f'{a}_minus_{b}', f'{a}_times_{b}']
    evaluate(x3, BASE_COLS + new_cols, CAT_COLS, f'E5: {a} & {b}')

### Conclusion — E5
Every pair gave a change between **−0.00008 and −0.00012**. **Rejected.**
Even the arithmetic of the two strongest ratings adds nothing: the tree already approximates these functions with a few splits
(each rating only has 5 values), so the new columns are redundant.

<a id="e6"></a>
## 12. E6 — Removing weak features

### Idea origin
*Feature-importance analysis (section 10).* Columns that the model almost never uses might only add noise.
We try to drop each of the three weakest columns, and all three together.

In [ ]:
candidates = ['arrival_delay_missing', 'gender', 'departure_delay_in_minutes']

experiments = {name: [name] for name in candidates}    # one column at a time
experiments['all three'] = candidates                  # and all three together

for name, to_drop in experiments.items():
    cols = [c for c in BASE_COLS if c not in to_drop]
    cat  = [c for c in CAT_COLS if c in cols]          # also remove it from the list of categorical columns

    evaluate(x3, cols, cat, f'E6: without {name}')

### Conclusion — E6
Every removal changed the score by **−0.00002 … −0.00007**, i.e. slightly worse or noise. **Keep all 21 columns.**
Low importance does not mean "useless": removing these columns did not make the model better.

<a id="e7"></a>
## 13. E7 — Ratings as categories

### Idea origin
*EDA observation.* Rating `0` behaves very differently from 1–5 (section 5), and the relation between rating and satisfaction
is not smooth (jump between 3 and 4). If ratings are treated as **labels** instead of numbers, LightGBM can give each
value its own effect (including `0`). We test two variants: all 13 ratings as categories, or only the four columns with many zeros.

In [ ]:
# Variant A: all 13 rating columns as categories
x_cat = x3[BASE_COLS].copy()

for col in RATING_COLS:
    x_cat[col] = x_cat[col].astype('category')

cat_all = CAT_COLS + RATING_COLS
score_all = evaluate(x_cat, list(x_cat.columns), cat_all, 'E7a: all 13 ratings as categories')

In [ ]:
# Variant B: only the four columns that contain many zeros
DIGITAL4 = ['inflight_wifi_service', 'online_boarding',
            'ease_of_online_booking', 'departure/arrival_time_convenient']

x_cat4 = x3[BASE_COLS].copy()

for col in DIGITAL4:
    x_cat4[col] = x_cat4[col].astype('category')

cat_4 = CAT_COLS + DIGITAL4
score_4 = evaluate(x_cat4, list(x_cat4.columns), cat_4, 'E7b: 4 columns as categories')

### Conclusion — E7
| Variant | Change vs baseline |
|---|---|
| all 13 as categories | −0.00013 |
| 4 columns as categories | −0.00004 |

**Rejected.** LightGBM already separates the value `0` from the others with its numeric splits; treating the ratings as
unordered labels throws away the natural order 1 < 2 < … < 5 and does not help.

<a id="e8"></a>
## 14. E8 — Frequency features (the idea that worked)

### Idea origin
After E1–E7 the pattern was clear: features built from *the same row* do not help. What can a tree **not** know?
**Information about other rows.** The simplest such information is *how often a value occurs in the data*
(frequency encoding).

Why `flight_distance`, `age` and `departure_delay_in_minutes`? They are numeric columns with many distinct values, so the
count of a value is informative (common vs rare). In a synthetic (generated) dataset, rare values can mean something
different than common ones.

Key details:
* The count is computed over **train + test together**. There is no target in this calculation, so **no leakage**.
* `value_counts()` gives "value → how many times"; `map()` writes it to every row.

In [ ]:
FREQ_COLS = ['flight_distance', 'age', 'departure_delay_in_minutes']

# Count over train and test together (no target is used here)
full_xy = pd.concat([X[FREQ_COLS], X_test[FREQ_COLS]])

for col in FREQ_COLS:
    freq = full_xy[col].value_counts()          # how many times each value occurs
    x3[f'{col}_freq'] = x3[col].map(freq)       # write the count of its value to every row

In [ ]:
freq_cols = [f'{c}_freq' for c in FREQ_COLS]
print(x3[freq_cols].describe())

# All three frequency features together
score_freq = evaluate(x3, BASE_COLS + freq_cols, CAT_COLS, 'E8a: three frequency features')

In [ ]:
# One at a time, to see which one carries the signal
for col in FREQ_COLS:
    f = f'{col}_freq'
    evaluate(x3, BASE_COLS + [f], CAT_COLS, f'E8b: {f}')

**Why does it work?** Split the rows into 10 groups by the frequency of their `flight_distance` and look at the share of
satisfied passengers in each group.

In [ ]:
freq_bins = pd.qcut(x3['flight_distance_freq'], 10, duplicates='drop')    # 10 groups of equal size
decile_stats = y.groupby(freq_bins).agg(['mean', 'count'])
print(decile_stats)

# Chart: the rarer the distance, the more satisfied the passengers
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(decile_stats) + 1), decile_stats['mean'].values, color='#2563eb', width=0.6)
for i, v in enumerate(decile_stats['mean'].values, 1):
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', color='#1f2937', fontsize=9)
ax.set_xlabel('Decile of flight_distance frequency (1 = rarest distances, 10 = most common)', color='#6b7280')
ax.set_ylabel('Share of satisfied', color='#6b7280')
ax.set_title('Rare flight distances go with more satisfied passengers', color='#1f2937', loc='left')
ax.grid(axis='y', color='#e5e7eb'); ax.set_axisbelow(True)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

### Conclusion — E8
| Feature | Change vs baseline |
|---|---|
| **`flight_distance_freq`** | **+0.00059** |
| `age_freq`, `departure_delay_in_minutes_freq` | −0.00005 (noise / harmful) |

**Accepted: `flight_distance_freq`** — the only feature so far above the signal threshold. The decile table shows the reason:
passengers with the rarest distances (frequency up to 121) are about 69% satisfied, while with the most common distances only
about 33–37% are. The model cannot see this by looking at one row, because it does not know how often that distance appears elsewhere.
The other two frequency features are dropped.

The new reference for later experiments is `REF_FREQ = 0.95944` (baseline + `flight_distance_freq`).

<a id="e9"></a>
## 15. E9 — Frequency of value combinations

### Idea origin
*Extension of E8.* If the count of one value is useful, the count of a **combination** may be even more specific:
"how many passengers have this distance **and** this type of travel?". We add `flight_distance` combined with each of
five other columns, one at a time, on top of `flight_distance_freq`.

Technique: glue the two values into one string key (`"1200_Business travel"`), count the keys, write the counts to each row.
`full` holds train and test one after another; the first `n` rows are train, so `iloc[:n]` returns the train part.

In [ ]:
REF_FREQ = 0.95944      # CV AUC with the single flight_distance_freq feature (E8)

combos = [
    ('flight_distance', 'class'),
    ('flight_distance', 'type_of_travel'),
    ('flight_distance', 'customer_type'),
    ('flight_distance', 'age'),
    ('flight_distance', 'gender'),
]

full = pd.concat([X, X_test], ignore_index=True)    # train + test one after another
n = len(X)                                          # number of train rows

for a, b in combos:
    key = full[a].astype(str) + '_' + full[b].astype(str)    # combination key
    counts = key.map(key.value_counts())                      # how often this key occurs
    x3[f'{a}_{b}_freq'] = counts.iloc[:n].values              # keep only the train rows

In [ ]:
for a, b in combos:
    new = f'{a}_{b}_freq'
    evaluate(x3, BASE_COLS + ['flight_distance_freq', new], CAT_COLS,
             f'E9a: + {new}', ref=REF_FREQ, ref_name='REF_FREQ')

Do larger combinations (three or four columns) or several pairs together give more? We build a small helper for
combinations of any length and test three variants.

In [ ]:
def freq_feature(cols):
    """Frequency of the combination of columns `cols` in train+test; returns values for the train rows."""
    key = full[cols[0]].astype(str)
    for c in cols[1:]:
        key = key + '_' + full[c].astype(str)
    counts = key.map(key.value_counts())
    return counts.iloc[:n].values

x3['fd_class_travel_freq']      = freq_feature(['flight_distance', 'class', 'type_of_travel'])
x3['fd_class_travel_cust_freq'] = freq_feature(['flight_distance', 'class', 'type_of_travel', 'customer_type'])

In [ ]:
experiments = {
    'both pairs together': ['flight_distance_freq',
                            'flight_distance_class_freq',
                            'flight_distance_type_of_travel_freq'],
    'triple':              ['flight_distance_freq', 'fd_class_travel_freq'],
    'quadruple':           ['flight_distance_freq', 'fd_class_travel_cust_freq'],
}

for name, extra in experiments.items():
    evaluate(x3, BASE_COLS + extra, CAT_COLS, f'E9b: {name}', ref=REF_FREQ, ref_name='REF_FREQ')

### Conclusion — E9
Change relative to `REF_FREQ` (baseline + `flight_distance_freq`):

| Added combination | Change vs REF_FREQ |
|---|---|
| `flight_distance` + `type_of_travel` | **+0.00020** |
| `flight_distance` + `class` | +0.00016 |
| `flight_distance` + `customer_type` | +0.00008 |
| `flight_distance` + `age` | 0.00000 |
| `flight_distance` + `gender` | 0.00000 |
| triple / quadruple / both pairs | +0.00020 / +0.00015 / +0.00010 (plateau, no better than the single pair) |

**Accepted: `flight_distance_type_of_travel_freq`.** The pair with `type_of_travel` is the best and the simplest; larger
combinations do not improve on it, so we stop here. The two accepted features together give **+0.00079** over the baseline.
`segment` from E4 is not part of the final set.

<a id="s16"></a>
## 16. Final feature set, verification and export

The final set = the 21 baseline features + 2 frequency features:
* `flight_distance_freq` — how many times this `flight_distance` appears in train + test.
* `flight_distance_type_of_travel_freq` — how many times this (`flight_distance`, `type_of_travel`) pair appears in train + test.

We wrap the logic in one function, `add_features()`, so the same code can be reused in the modelling notebook
(and in the same way on train and test).

> **Note.** This is the final feature set of **round 1** (CV 0.95964). Round 2 (section 18) adds one more feature, a target encoding that has to be computed **inside each CV fold**, so it cannot be stored in the saved CSV files.

In [ ]:
def add_features(train_df, test_df):
    """Add the two frequency features. Counts are made over train+test together (there is no target here)."""
    train_df = train_df.copy()
    test_df  = test_df.copy()

    full = pd.concat([train_df, test_df], ignore_index=True)    # train rows first, then test rows
    n = len(train_df)

    # 1. Frequency of the flight distance value
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values   # first n rows = train
    test_df['flight_distance_freq']  = counts.iloc[n:].values   # the rest = test

    # 2. Frequency of the (distance, type of travel) combination
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values

    return train_df, test_df

In [ ]:
X_fe, X_test_fe = add_features(X, X_test)

# Check 1: shapes
print(X_fe.shape, X_test_fe.shape)       # expected (699635, 23) (299844, 23)

# Check 2: the function gives the same numbers as our step-by-step experiments (both must be True)
print(np.allclose(X_fe['flight_distance_freq'], x3['flight_distance_freq']))
print(np.allclose(X_fe['flight_distance_type_of_travel_freq'],
                  x3['flight_distance_type_of_travel_freq']))

In [ ]:
FINAL_CV_RECORDED = 0.95964      # recorded result of this cell

score_fe, oof_fe, test_pred_fe = run_cv(X_fe, y, X_test_fe, CAT_COLS)
print(f"CV with add_features: {score_fe:.5f}  gain vs baseline {score_fe - BASELINE_CV:+.5f}")

**Export** for the modelling notebook. Note: CSV files do not store the pandas `category` type, so the four text columns
(`gender`, `customer_type`, `type_of_travel`, `class`) must be converted to `category` again after loading
(for CatBoost, plain strings are enough).

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)

X_fe.to_csv(PROC / 'X_train.csv', index=False)
X_test_fe.to_csv(PROC / 'X_test.csv', index=False)
y.to_csv(PROC / 'y_train.csv', index=False)

print(X_fe.shape, X_test_fe.shape, y.shape)

### Conclusion — final set
CV AUC **0.95964** versus the baseline **0.95885** (**+0.00079**), 23 features in total. The files `X_train.csv`,
`X_test.csv` and `y_train.csv` in `data/processed/` are ready for the modelling stage.

<a id="s17"></a>
## 17. Using the original dataset

### Idea origin
*Standard technique for Playground competitions.* Playground data is generated from a real dataset
([Kaggle: binary-aviation-satisfaction-129k](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k),
129,880 rows). Sometimes the original rows or a model trained on them help. This section needs the file
`orig_data.csv` in the raw-data folder. It is a **side branch**: it does not change the final feature set.

Plan:
1. Compare the original with train (structure, target rate, distances, exact duplicates).
2. Use it in two ways: (a) predictions of a model trained on the original as a new feature; (b) original rows added to the training part of each fold.

In [ ]:
orig = pd.read_csv(DATA_DIR / 'orig_data.csv')
orig.columns = orig.columns.str.lower().str.replace(' ', '_')

print(orig.shape)
print(orig.columns.tolist())
print(orig.head(3))

# Which column names differ between our train and the original?
print(set(train.columns) ^ set(orig.columns))

### 17.1 Do the two datasets look alike?
Check whether the flight distances from our train/test exist in the original, and compare the target rates and typical frequencies.
This tests the guess "rare distances in train are values that never occur in the original".

In [ ]:
orig_dist = set(orig['flight_distance'])                    # distances present in the original

in_orig = train['flight_distance'].isin(orig_dist)          # does the row's distance occur in the original?

print("share satisfied: train", round(train['satisfaction'].mean(), 3),
      "| original", round(orig['satisfaction'].mean(), 3))
print("share of train rows whose distance is in the original:", round(in_orig.mean(), 3))
print("share of test rows whose distance is in the original: ", round(test['flight_distance'].isin(orig_dist).mean(), 3))

print(y.groupby(in_orig).agg(['mean', 'count']))                         # satisfied rate: in / not in the original
print(x3['flight_distance_freq'].groupby(in_orig).median())              # typical frequency: in / not in the original

### 17.2 Are there exact copies of original rows in train?
We merge on all columns except the target. If copies existed, we could simply copy the original label.
`drop_duplicates` keeps one original row per unique combination so the merge does not multiply rows.

In [ ]:
match_cols = [c for c in orig.columns if c not in ['satisfaction', 'arrival_delay_in_minutes']]

orig_u = orig.drop_duplicates(subset=match_cols)              # one row per unique combination
merged = train.merge(orig_u[match_cols + ['satisfaction']],
                     on=match_cols, how='left', suffixes=('', '_orig'))

matched = merged['satisfaction_orig'].notna()                 # train rows that found a copy in the original

print("rows in train:", len(train), "| after merge:", len(merged))   # must be equal
print("share of train rows with an exact match in the original:", round(matched.mean(), 4))
print("original label equals train label:",
      round((merged.loc[matched, 'satisfaction'] == merged.loc[matched, 'satisfaction_orig']).mean(), 4))

### 17.3 A model trained only on the original data
We bring the original to the same format as `X` (same columns, same order, same category codes), train LightGBM
on it and score it on our train. A high AUC means the two datasets follow the same rules.

In [ ]:
# Bring the original to the same shape as X
orig_X = orig.drop(columns=['satisfaction', 'arrival_delay_in_minutes']).copy()
orig_X['arrival_delay_missing'] = orig['arrival_delay_in_minutes'].isna().astype(int)
orig_X = orig_X[X.columns]                     # same columns in the same order
orig_y = orig['satisfaction'].astype(int)

# Categories with the same codes as in X
for col in CAT_COLS:
    orig_X[col] = pd.Categorical(orig_X[col], categories=X[col].cat.categories)

params = {**LGBM_PARAMS, 'n_estimators': 300, 'early_stopping_rounds': None}
m_orig = lgb.LGBMClassifier(**params)
m_orig.fit(orig_X, orig_y, categorical_feature=CAT_COLS)

pred_on_train = m_orig.predict_proba(X)[:, 1]
print("AUC of the original-data model on our train:", round(roc_auc_score(y, pred_on_train), 5))

### 17.4 Way (a): prediction of the original-data model as a feature (`orig_pred`)
The prediction is made by a model that was **not** trained on our train rows, so it contains no information about our
target — no leakage.

In [ ]:
# Prediction of the original-data model as a new feature
X_d = X.copy()
X_d['orig_pred'] = m_orig.predict_proba(X)[:, 1]

X_test_d = X_test.copy()
X_test_d['orig_pred'] = m_orig.predict_proba(X_test)[:, 1]

score_d, _, _ = run_cv(X_d, y, X_test_d, CAT_COLS)
print(f"+ orig_pred: {score_d:.5f}  gain vs baseline {score_d - BASELINE_CV:+.5f}\n")

In [ ]:
# The same on top of the frequency features (reference: 0.95964)
X_fd = X_fe.copy()
X_fd['orig_pred'] = X_d['orig_pred']

X_test_fd = X_test_fe.copy()
X_test_fd['orig_pred'] = X_test_d['orig_pred']

score_fd, _, _ = run_cv(X_fd, y, X_test_fd, CAT_COLS)
print(f"+ frequencies + orig_pred: {score_fd:.5f}  vs baseline {score_fd - BASELINE_CV:+.5f}  vs frequencies {score_fd - FINAL_CV_RECORDED:+.5f}")

### 17.5 Way (b): original rows added to the training part of every fold
`run_cv_aug` is `run_cv` with one change: in each fold the original rows are **appended to the training part only**.
Validation uses competition rows only, so the score stays comparable. Optionally an `is_original` flag (0 for competition rows,
1 for original rows) lets the model tell the two sources apart.

In [ ]:
def run_cv_aug(X, y, X_test, cat_cols, extra_X, extra_y, add_flag=False):
    """CV where the rows extra_X (the original data) are added to the training part of every fold.
    Validation is done on competition rows only."""
    if add_flag:
        X       = X.copy();       X['is_original'] = 0
        X_test  = X_test.copy();  X_test['is_original'] = 0
        extra_X = extra_X.copy(); extra_X['is_original'] = 1

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof_preds   = np.zeros(len(X))
    test_preds  = np.zeros(len(X_test))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        # training part: a part of train + ALL original rows
        X_tr = pd.concat([X.iloc[tr_idx], extra_X], ignore_index=True)
        y_tr = pd.concat([y.iloc[tr_idx], extra_y], ignore_index=True)
        # validation part: competition rows only
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**LGBM_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=cat_cols)

        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"Fold {fold} AUC: {score:.5f}")

    cv_score = roc_auc_score(y, oof_preds)
    print(f"\nCV AUC (OOF): {cv_score:.5f}")
    print(f"Mean fold AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")
    return cv_score, oof_preds, test_preds

For the variants that use the frequency features, the original rows need the same two columns. We compute their counts
with the **train+test** frequency tables, so a value has the same count in every table.

In [ ]:
full = pd.concat([X, X_test], ignore_index=True)

orig_fe = orig_X.copy()
orig_fe['flight_distance_freq'] = orig_fe['flight_distance'].map(full['flight_distance'].value_counts())

key_full = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
key_orig = orig_fe['flight_distance'].astype(str) + '_' + orig_fe['type_of_travel'].astype(str)
orig_fe['flight_distance_type_of_travel_freq'] = key_orig.map(key_full.value_counts())

orig_fe = orig_fe[X_fe.columns]
print(orig_fe[['flight_distance_freq', 'flight_distance_type_of_travel_freq']].isna().sum())   # expected 0

In [ ]:
experiments = [
    # name,                          X,    X_test,    extra rows, flag,  reference score
    ('21 columns + original',        X,    X_test,    orig_X,  False, BASELINE_CV),
    ('21 columns + original + flag', X,    X_test,    orig_X,  True,  BASELINE_CV),
    ('frequencies + original',       X_fe, X_test_fe, orig_fe, False, FINAL_CV_RECORDED),
    ('frequencies + original + flag', X_fe, X_test_fe, orig_fe, True, FINAL_CV_RECORDED),
]

for name, Xa, Xta, extra, flag, ref in experiments:
    score, _, _ = run_cv_aug(Xa, y, Xta, CAT_COLS, extra, orig_y, add_flag=flag)
    print(f">>> {name}: {score:.5f}  vs reference {score - ref:+.5f}\n")

### Conclusion — original dataset
* **Similar rules, no copies.** Target rate: 0.434 (original) vs 0.444 (train). There are no exact copies of original rows in
  train. A model trained only on the original reaches AUC **0.95436** on our train, so the data follow almost the same logic.
* **All distances exist in the original**, so rare distances are not "values that never occur in the original".
* **Way (a), `orig_pred`:** −0.00008 vs baseline; on top of the frequencies −0.00018 vs 0.95964. No gain.
* **Way (b), appended rows:** without the flag **−0.00027** (21 columns) and **−0.00035** (with frequencies); with the `is_original` flag
  −0.00003 and −0.00008 (noise). Mixing the sources without a flag hurts because their distributions differ slightly.

**Decision: the original dataset is not used in the final pipeline.**

<a id="s18"></a>
## 18. Round 2 — ideas from public notebooks

After round 1 (final CV **0.95964**) we studied two public notebooks: a LightGBM notebook (target encoding, digit features,
distance profile) and a RealMLP + XGBoost stack. They confirm that the frequency features of `flight_distance` matter and suggest more variants.
Each idea is tested with the **same protocol** as before, now against the reference `REF_FE = 0.95964`
(for ideas built on top of target encoding: `REF_TE = 0.96015`).

**What to run before this section:** sections 1–3 (setup, preprocessing, `run_cv`) and the `add_features` cell of section 16 (it creates `X_fe` and `X_test_fe`).
Experiments in this section compare **only against the reference**, they do not depend on E1–E9.

**Lessons we carry in:** a column derived from the *same* column rarely helps a tree (E1–E7); information about *other rows* does (E8).
Both lessons are tested again below.

In [ ]:
REF_FE = 0.95964          # recorded CV of the final set of round 1 (21 baseline + 2 frequency features)

X_fe, X_test_fe = add_features(X, X_test)     # the 23-feature set of round 1
x4 = X_fe.copy()                              # working copy for round 2, X_fe stays untouched
print(X_fe.shape, X_test_fe.shape)            # expected (699635, 23) (299844, 23)

<a id="e10"></a>
### E10 — Views of `flight_distance` (bins)

**Idea origin:** public notebook. The frequency of the exact distance worked (E8), so coarser views may show the same effect at another scale:
`// 10` and `// 100` (groups of neighbouring distances) and `% 1000` (the "tail" of the number, where generation artifacts may hide).

In [ ]:
x4['fd_div10']   = x4['flight_distance'] // 10
x4['fd_div100']  = x4['flight_distance'] // 100
x4['fd_mod1000'] = x4['flight_distance'] % 1000

for new_col in ['fd_div10', 'fd_div100', 'fd_mod1000']:
    evaluate(x4, list(X_fe.columns) + [new_col], CAT_COLS,
             f'E10: + {new_col}', ref=REF_FE, ref_name='REF_FE')

**Conclusion — E10**

| Variant | Change vs REF_FE | Sign over the 5 folds |
|---|---|---|
| `fd_div10` | −0.00011 | all negative |
| `fd_div100` | −0.00012 | 4 of 5 negative |
| `fd_mod1000` | −0.00001 | mixed |

**Rejected.** Coarse bins add no information: a tree can split `flight_distance` by thresholds itself and reproduces such groups. The extra columns only add noise.
`% 1000` gave zero with mixed signs, so there is no hidden pattern in the "tail" of the distance.
*Lesson (same as E1–E7):* a new column made from the same column does not help a tree.

<a id="e11"></a>
### E11 — Digits of numbers

**Idea origin:** public notebook (claimed +0.00069, but mixed with other changes in the same step). The idea: if the synthetic generator rounded values in some way,
the last digit of a number could carry a signal that a tree cannot see (a tree splits by thresholds, not by "last digit").

First check: are the columns integers? Then check the link between the **last digit** and the target *without* training a model.

In [ ]:
# 1. Do the numeric columns have decimals? (if not, only the positions 0..3 of a number are real digits)
for col in ['age', 'flight_distance', 'departure_delay_in_minutes']:
    frac = (X_fe[col] % 1 != 0).mean()
    print(col, '| dtype:', X_fe[col].dtype, '| share with decimals:', round(frac, 4),
          '| unique values:', X_fe[col].nunique())

# 2. Share of satisfied passengers by the last digit (about 70,000 rows per digit -> random noise is about +-0.002)
for col in ['flight_distance', 'age', 'departure_delay_in_minutes']:
    last_digit = X_fe[col] % 10
    rate = y.groupby(last_digit).mean()
    print(col, '| spread (max - min):', round(rate.max() - rate.min(), 4))

All three columns are integers. The spread of the satisfied share between last digits is **0.10** for `flight_distance`, 0.067 for `age`, 0.075 for `departure_delay`
(random noise would be about 0.002), so the link looks real. But a link with the target does **not** prove that the model gets new information, so we test it with CV.

In [ ]:
x4['fd_d0']    = x4['flight_distance'] % 10                  # last digit of the distance
x4['fd_d1']    = (x4['flight_distance'] // 10) % 10          # tens digit of the distance
x4['age_d0']   = x4['age'] % 10
x4['delay_d0'] = x4['departure_delay_in_minutes'] % 10

for new_col in ['fd_d0', 'fd_d1', 'age_d0', 'delay_d0']:
    evaluate(x4, list(X_fe.columns) + [new_col], CAT_COLS,
             f'E11: + {new_col}', ref=REF_FE, ref_name='REF_FE')

**Conclusion — E11**

| Variant | Change vs REF_FE | Sign over the 5 folds |
|---|---|---|
| `fd_d0` (last digit of distance) | −0.00003 | mixed |
| `fd_d1` (tens digit of distance) | +0.00006 | 4 of 5 positive, tiny |
| `age_d0` | **−0.00022** | all negative |
| `delay_d0` | −0.00012 | all negative |

**Rejected.** The last digit of the distance has a strong link with the target (spread 0.10), yet it adds nothing in CV. Most likely the model already has this information
through the raw distance and `flight_distance_freq`.

*Lesson:* **an association with the target is not the same as new information for the model.** Always test with CV.

<a id="e12"></a>
### E12 — Target encoding of `flight_distance` (accepted)

**Idea origin:** public notebook. `flight_distance` has 3,474 distinct values (about 200 rows each), so a tree cannot learn the behaviour of each value from splits.
**Target encoding (TE)** replaces a value by the share of satisfied passengers among all passengers with that value, and gives the model this number directly.
(Frequency says *how often* a value occurs, TE says *how satisfied* those passengers are.)

**The danger is leakage.** If TE is computed on the whole train set, a row sees its own target inside the average. Training looks great, validation does not.
Therefore, **inside every CV fold**:
1. fit the encoder on the **training part** of the fold only;
2. for the training rows use **internal cross-fitting** (`cv=5` in `TargetEncoder`), so a row never sees its own target;
3. for validation and test rows use the statistics of the whole training part.

**Smoothing (`smooth`)**: a value with 3 rows, all satisfied, would get TE = 1.0 which is just chance. Smoothing pulls such estimates towards the global rate:
`TE = (n · rate_of_value + m · global_rate) / (n + m)`, where `n` is the number of rows of the value and `m` is the smoothing strength (`'auto'` picks it from the data).
Smoothing matters most for rare values; with about 200 rows per value it should matter little (we check 10 and 100 too).

`run_cv_te3` is `run_cv` plus the encoding step. Keys are given as small functions; a key goes only to the encoder, the model receives the number `te_<name>`.

In [ ]:
from sklearn.preprocessing import TargetEncoder


def run_cv_te3(X, y, X_test, cat_cols, te_keys, smooth='auto'):
    """Like run_cv, but adds target-encoded features INSIDE each fold (no leakage).
    te_keys is a dict {feature name: function(df) -> key Series}. The key only goes to the encoder,
    the model receives the encoded number te_<name>."""
    names = list(te_keys.keys())
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof_preds, test_preds, fold_scores = np.zeros(len(X)), np.zeros(len(X_test)), []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr,  X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_tr,  y_val = y.iloc[train_idx], y.iloc[val_idx]
        X_te = X_test.copy()

        keys_tr  = pd.DataFrame({n: te_keys[n](X_tr)  for n in names})
        keys_val = pd.DataFrame({n: te_keys[n](X_val) for n in names})
        keys_te  = pd.DataFrame({n: te_keys[n](X_te)  for n in names})

        enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=42)
        enc_tr  = enc.fit_transform(keys_tr, y_tr)       # cross-fitted inside the training part
        enc_val = enc.transform(keys_val)                # statistics of the whole training part
        enc_te  = enc.transform(keys_te)

        for i, n in enumerate(names):
            X_tr[f'te_{n}']  = enc_tr[:, i]
            X_val[f'te_{n}'] = enc_val[:, i]
            X_te[f'te_{n}']  = enc_te[:, i]

        model = lgb.LGBMClassifier(**LGBM_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=cat_cols)

        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        test_preds += model.predict_proba(X_te)[:, 1] / N_FOLDS

        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"Fold {fold} AUC: {score:.5f}")

    cv_score = roc_auc_score(y, oof_preds)
    print(f"\nCV AUC (OOF): {cv_score:.5f}")
    print(f"Mean fold AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")
    return cv_score, oof_preds, test_preds


# How each key is built from a table
KEY_FD = lambda d: d['flight_distance'].astype(str)          # exact distance

In [ ]:
# E12a: TE of the exact distance, smoothing chosen automatically
score, _, _ = run_cv_te3(X_fe, y, X_test_fe, CAT_COLS, te_keys={'flight_distance': KEY_FD}, smooth='auto')
print(f">>> E12: TE flight_distance (auto): CV AUC {score:.5f} | vs REF_FE {score - REF_FE:+.5f}")

In [ ]:
# E12b: how sensitive is the result to the smoothing strength?
for s in [10.0, 100.0]:
    score, _, _ = run_cv_te3(X_fe, y, X_test_fe, CAT_COLS, te_keys={'flight_distance': KEY_FD}, smooth=s)
    print(f">>> E12: TE flight_distance (smooth={s}): CV AUC {score:.5f} | vs REF_FE {score - REF_FE:+.5f}\n")

**Conclusion — E12**

| Variant | CV AUC | Change vs REF_FE | Sign over the 5 folds |
|---|---|---|---|
| `smooth='auto'` | 0.96015 | **+0.00051** | all 5 positive (+0.0004 … +0.0009) |
| `smooth=10` | 0.96018 | +0.00054 | all 5 positive |
| `smooth=100` | 0.96015 | +0.00051 | all 5 positive |

**Accepted: `te_flight_distance` with `smooth='auto'`.** It is above the +0.0003 threshold and positive on every fold. No leakage: the CV stays in the normal range (it would be 0.97+ with a leak).
The result hardly depends on the smoothing (differences of 0.00003 are noise) because every distance has about 200 rows, so there are almost no very rare values.

New reference for the next experiments: `REF_TE = 0.96015`.

<a id="e13"></a>
### E13 — More columns in target encoding

**Idea origin:** extension of E12 and of E9 (the pair distance × `type_of_travel` helped the frequency features). We add one more TE feature at a time on top of the accepted TE of the distance:
`age`, distance × `type_of_travel`, distance × `class`. A pair has more, smaller groups (about 100 rows each), so smoothing matters more here.

In [ ]:
REF_TE = 0.96015          # recorded CV of E12 (frequency set + TE of flight_distance)

KEY_AGE       = lambda d: d['age'].astype(str)
KEY_FD_TYPE   = lambda d: d['flight_distance'].astype(str) + '_' + d['type_of_travel'].astype(str)
KEY_FD_CLASS  = lambda d: d['flight_distance'].astype(str) + '_' + d['class'].astype(str)

experiments = {
    '+ age':                       {'fd': KEY_FD, 'age': KEY_AGE},
    '+ distance x type_of_travel': {'fd': KEY_FD, 'fd_type': KEY_FD_TYPE},
    '+ distance x class':          {'fd': KEY_FD, 'fd_class': KEY_FD_CLASS},
}

for name, keys in experiments.items():
    score, _, _ = run_cv_te3(X_fe, y, X_test_fe, CAT_COLS, te_keys=keys)
    print(f">>> E13: TE flight_distance {name}: CV AUC {score:.5f} | vs REF_TE {score - REF_TE:+.5f} | vs REF_FE {score - REF_FE:+.5f}\n")

**Conclusion — E13**

| Added to the TE of the distance | CV AUC | Change vs REF_TE | Sign over the 5 folds |
|---|---|---|---|
| `age` | 0.96020 | +0.00005 | 4 of 5 positive, tiny |
| distance × `type_of_travel` | 0.96016 | +0.00001 | mixed |
| distance × `class` | 0.96026 | +0.00011 | all 5 positive, but tiny |

**Not taken.** The best variant (+0.00011) is at the noise level and far below +0.0003 (the same rule as for `segment` in E4). The TE of the distance already captures the signal;
this is a plateau, like E9. The pair with `type_of_travel` helped frequencies, but gives nothing for TE: the model already has this information.

<a id="e14"></a>
### E14 — Distance profile

**Idea origin:** public notebook ("distance profile"). Information about *other* rows, like the frequency: for each distance, the average rating of all passengers who flew that distance
("a typical passenger on this route") and the deviation of the passenger from it. It is computed **without the target** on train + test together, so there is no leakage.
We start with the three strongest ratings (see section 10): 6 new columns.

In [ ]:
PROFILE_COLS = ['online_boarding', 'inflight_wifi_service', 'inflight_entertainment']

# Statistics per flight_distance on train and test together (no target is used)
full = pd.concat([X_fe, X_test_fe], ignore_index=True)
n = len(X_fe)

prof = pd.DataFrame(index=full.index)
for col in PROFILE_COLS:
    mean = full.groupby('flight_distance')[col].transform('mean')   # typical rating on this distance
    prof[f'fd_mean_{col}']  = mean
    prof[f'fd_resid_{col}'] = full[col] - mean                       # passenger vs the typical passenger

# Split back: the first n rows are train, the rest are test
X_c      = pd.concat([X_fe,      prof.iloc[:n].reset_index(drop=True)], axis=1)
X_test_c = pd.concat([X_test_fe, prof.iloc[n:].reset_index(drop=True)], axis=1)
print(X_c.shape, X_test_c.shape)       # expected (699635, 29) (299844, 29)

score, _, _ = run_cv_te3(X_c, y, X_test_c, CAT_COLS, te_keys={'flight_distance': KEY_FD})
print(f">>> E14: profile of 3 ratings + TE: CV AUC {score:.5f} | vs REF_TE {score - REF_TE:+.5f}")

**Conclusion — E14**

CV AUC 0.96015, change vs REF_TE **±0.00000**, mixed signs over the folds. **Rejected.** About 200 passengers per distance is too few for a stable "typical rating", and the distances of a synthetic dataset
are probably not real routes. Whatever signal there is in the distance is already taken by the TE. The full variant (about 60 columns) was not tried: if the six strongest columns give nothing, the rest would only add noise.

<a id="e15"></a>
### E15 — Target encoding of neighbourhoods

**Idea origin:** public notebook (TE of binned distance and age). The exact TE uses only passengers with *exactly* the same distance. Distances 1230 and 1240 are probably similar,
so a TE by bins (groups of neighbouring distances) is more stable and may catch a smooth link. The bins go **only to the encoder** (E10 showed that bins as model columns do not help).

In [ ]:
KEY_FD_10   = lambda d: (d['flight_distance'] // 10).astype(str)       # neighbourhood: groups of 10
KEY_FD_100  = lambda d: (d['flight_distance'] // 100).astype(str)      # groups of 100
KEY_FD_250  = lambda d: (d['flight_distance'] // 250).astype(str)      # groups of 250
KEY_AGE_5   = lambda d: (d['age'] // 5).astype(str)                    # age in 5-year groups

experiments = {
    '+ distance // 10':  {'fd': KEY_FD, 'fd_10':  KEY_FD_10},
    '+ distance // 100': {'fd': KEY_FD, 'fd_100': KEY_FD_100},
    '+ distance // 250': {'fd': KEY_FD, 'fd_250': KEY_FD_250},
    '+ age // 5':        {'fd': KEY_FD, 'age_5':  KEY_AGE_5},
}

for name, keys in experiments.items():
    score, _, _ = run_cv_te3(X_fe, y, X_test_fe, CAT_COLS, te_keys=keys)
    print(f">>> E15: TE flight_distance {name}: CV AUC {score:.5f} | vs REF_TE {score - REF_TE:+.5f} | vs REF_FE {score - REF_FE:+.5f}\n")

**Conclusion — E15**

| Added to the TE of the distance | CV AUC | Change vs REF_TE |
|---|---|---|
| `distance // 10` | 0.96017 | +0.00002 |
| `distance // 100` | 0.96016 | +0.00001 |
| `distance // 250` | 0.96015 | −0.00000 |
| `age // 5` | 0.96009 | −0.00006 |

**Rejected.** The TE of the exact distance already takes the signal along this axis; neighbourhoods add nothing. This closes the family of ideas around `flight_distance`.

### Final feature set after round 2

**21 baseline features + 2 frequency features (round 1) + `te_flight_distance`.** CV AUC **0.96015** (+0.00130 versus the baseline 0.95885).

The target encoding must be computed **inside every fold**, so it is not part of the saved CSV files (they hold the 23 features of round 1).
`add_te_flight_distance` is a small reusable function for any training loop (LightGBM, CatBoost, XGBoost): call it inside each fold, with the training part of that fold as `X_fit`.
`run_cv_final` shows the usage and must reproduce the CV of E12.

In [ ]:
def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    """Add the column te_flight_distance. Call it INSIDE a fold:
    X_fit / y_fit = the training part of the fold, other_frames = list of frames to encode with the fitted encoder
    (validation part, test). Returns the encoded X_fit and the list of encoded other frames."""
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=42)
    key = lambda d: d[['flight_distance']].astype(str)                  # the key goes only to the encoder

    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]   # cross-fitted on the training part
    encoded = []
    for frame in other_frames:
        frame = frame.copy()
        frame['te_flight_distance'] = enc.transform(key(frame))[:, 0]
        encoded.append(frame)
    return X_fit, encoded


def run_cv_final(X, y, X_test, cat_cols):
    """The final pipeline of the feature stage: 5-fold CV with the in-fold target encoding."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof_preds, test_preds = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        X_tr, (X_val, X_te) = add_te_flight_distance(X_tr, y_tr, [X_val, X_test])

        model = lgb.LGBMClassifier(**LGBM_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=cat_cols)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1] / N_FOLDS
        print(f"Fold {fold} AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.5f}")

    cv_score = roc_auc_score(y, oof_preds)
    print(f"\nCV AUC (OOF): {cv_score:.5f}")
    return cv_score, oof_preds, test_preds


score_final, oof_final, test_pred_final = run_cv_final(X_fe, y, X_test_fe, CAT_COLS)
print(f"Final CV: {score_final:.5f} | vs baseline {score_final - BASELINE_CV:+.5f} | vs REF_TE {score_final - REF_TE:+.5f}")   # expected about 0.96015

**Conclusion — round 2.** One accepted feature (`te_flight_distance`, +0.00051 on top of round 1); all other ideas from the public notebooks gave nothing in our protocol.
The final CV is **0.96015**.

<a id="s19"></a>
## 19. Summary

### 19.1 Results of all experiments (change in CV AUC vs the reference, LightGBM)

In [ ]:
# All values are the results recorded when the experiments were run (hard-coded on purpose, so the chart is stable)
summary = [
    # (label,                                         change,   accepted?)
    ('E1 rating aggregates (all 7)',                 -0.00021, False),
    ('E2 delay_diff',                                -0.00001, False),
    ('E3 class-relative online_boarding',            -0.00003, False),
    ('E4 segment',                                   +0.00008, False),
    ('E5 arithmetic of top ratings (worst pair)',    -0.00012, False),
    ('E6 removing weak columns (worst case)',        -0.00007, False),
    ('E7 ratings as categories (all 13)',            -0.00013, False),
    ('E8 flight_distance_freq',                      +0.00059, True),
    ('E9 + distance x type_of_travel (vs E8)',       +0.00020, True),
    ('17 orig_pred',                                 -0.00008, False),
    ('17 original rows added, no flag',              -0.00027, False),
    ('E10 bins of distance (worst)',                 -0.00012, False),
    ('E11 digits of numbers (worst: age digit)',     -0.00022, False),
    ('E12 TE of flight_distance (vs round 1)',       +0.00051, True),
    ('E13 more TE columns (vs E12, best)',           +0.00011, False),
    ('E14 distance profile (vs E12)',                +0.00000, False),
    ('E15 TE of neighbourhoods (vs E12, worst)',     -0.00006, False),
]

labels  = [s[0] for s in summary][::-1]
values  = [s[1] for s in summary][::-1]
colors  = ['#2563eb' if s[2] else '#d9822b' for s in summary][::-1]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(labels, values, color=colors, height=0.6)
ax.axvspan(-0.0001, 0.0001, color='#e5e7eb', alpha=0.8, zorder=0)       # the noise zone
ax.axvline(0.0003, color='#6b7280', linestyle='--', linewidth=1)          # the signal threshold
ax.text(0.0003, len(labels) - 0.4, ' signal threshold +0.0003', color='#6b7280', fontsize=8, va='bottom')
for i, v in enumerate(values):
    ax.text(v + (0.00002 if v >= 0 else -0.00002), i, f'{v:+.5f}', va='center',
            ha='left' if v >= 0 else 'right', color='#1f2937', fontsize=8)
ax.set_xlim(-0.0005, 0.0009)
ax.set_xlabel('Change in CV AUC (grey band = noise, |change| < 0.0001)', color='#6b7280')
ax.set_title('Only frequency and target-encoding features of the distance clear the threshold', color='#1f2937', loc='left')
ax.grid(axis='x', color='#e5e7eb'); ax.set_axisbelow(True)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2563eb', label='accepted'), Patch(color='#d9822b', label='rejected / not taken')],
          frameon=False, loc='lower right')
plt.tight_layout(); plt.show()

### 19.2 What we ended with
* **Final feature set:** the 21 baseline columns + `flight_distance_freq` + `flight_distance_type_of_travel_freq` (round 1, 23 columns) + `te_flight_distance` (round 2, computed inside each fold).
* **CV AUC:** 0.95885 → 0.95964 (round 1, +0.00079) → **0.96015** (round 2, **+0.00130** in total), LightGBM, 5-fold `StratifiedKFold(random_state=42)`.
* **Files:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv` hold the **23 features of round 1**. The target encoding is added inside the CV loop by `add_te_flight_distance()` (section 18).

### 19.3 Main takeaways
1. **LightGBM extracts almost everything from the raw columns.** Row-wise summaries, differences, ratios, class-relative values, segments, bins and digits of the same column all failed to beat the noise.
2. **What worked carries information about other rows:** how often a value appears (frequency) and how satisfied the passengers with this value are (target encoding). A tree cannot compute either from one row.
3. **All the signal sits on one axis:** `flight_distance` (a high-cardinality column). Once frequency and TE of the exact value are in, pairs, neighbourhoods and profiles add nothing.
4. **Association is not new information.** The last digit of the distance had a spread of 0.10 in the target rate, yet gave nothing in CV. Always test with CV.
5. **Target encoding needs care:** compute it inside each fold, with cross-fitting on the training part, otherwise the CV is inflated by leakage.
6. **Rating `0` is not a missing or worst score**; it probably means "service not used".
7. **Measure carefully:** compare with a reference recomputed in the same conditions, respect the noise level (≈ 0.0001) and look at the sign over the folds.
8. **External data needs a source flag.** Appending the original dataset without `is_original` made the CV worse.
9. **Signal is concentrated** in a few columns (`online_boarding`, `inflight_wifi_service`, `class`, `type_of_travel`, `inflight_entertainment` ≈ 81% of gain).

### 19.4 Not tested — ideas for the modelling stage
* **CatBoost:** pass `flight_distance` as an extra categorical column (CatBoost builds its own target statistics, an analogue of our TE), or use `add_te_flight_distance`. The numeric frequency features can be used as they are.
* **XGBoost:** needs `enable_categorical=True` or encoding of the four text columns. **Linear models / neural networks:** need scaling and another encoding.
* Hyperparameter tuning (Optuna gave about +0.0004 for LightGBM on the 23 features of round 1), then ensembling on the **same folds**.
* **Seen in public notebooks, not tested here:** window target rates (smoothed satisfied share in numeric neighbourhoods), auxiliary-task features for neural networks (predict each rating from the other columns), rarity `-log(freq)` (equal to the frequency for trees, useful for neural networks), several smoothing levels of TE at once.
* Not checked: EDA hypotheses "Business + long flight", "young + personal travel", "loyal + delay" (expectation: low).